## Service types — ClusterIP, NodePort, LoadBalancer, ExternalName

`spec.type` controls *how reachable* the Service is. Four values, each a superset of the last (mostly).

**`ClusterIP`** *(default)* — a virtual IP from the service CIDR, **reachable only inside the cluster**. What every internal microservice should be.

**`NodePort`** — reachable on **every node's IP** at a port in `30000–32767`. Includes a ClusterIP. From outside: `http://<any-node-ip>:31234`. Use for **dev clusters** and **bare metal without a load balancer** — in real clouds you rarely expose it directly.

```yaml
spec:
  type: NodePort
  ports: [{ port: 80, targetPort: 5678, nodePort: 31234 }]
```

**`LoadBalancer`** — exposed through a **cloud load balancer** (implies a NodePort too). The cloud-controller-manager provisions the LB (AWS ELB, GCP, Azure) and writes its public IP into `status`. On `kind`/`minikube` there's no cloud, so these stay `Pending` unless a tool like `metallb` provides a fake LB.

**`ExternalName`** — a different beast: **no proxy, no selector, no endpoints** — just a DNS CNAME from the Service name to an external host.

```yaml
spec: { type: ExternalName, externalName: db.legacy.example.com }
```

| Type | Reachable from | Use for |
|---|---|---|
| ClusterIP | inside cluster | every internal microservice |
| NodePort | + every node's IP | dev, bare metal, behind an LB |
| LoadBalancer | + cloud LB IP | public services in cloud |
| ExternalName | DNS only | aliasing an external host |

On our map all four are the same **Service** chip dialed to different reach — the internal VIP is the base, each higher type just adds an outward door.